# Columbia University; MATH5380; Multi-Asset Portfolio Management 

## Clarity Kathryn Kummer CKK2129
#### Draft Code 1

In [33]:
import pandas as pd
import openpyxl
from dateutil.relativedelta import relativedelta


df = pd.read_excel('Final_Project2_Data.xlsx')
assert df.isnull().sum().sum() == 0, "DataFrame contains missing values"
#print(df.dtypes)
#df.head()
returns = df.copy()

returns = returns.rename(columns = {'Total returns in local currency': 'Date'}).set_index('Date').sort_index()
returns.head()

,MSCI Australia NR LCL,MSCI Canada NR LCL,MSCI France NR LCL,MSCI Germany NR LCL,MSCI Italy NR LCL,MSCI Japan NR JPY,MSCI Netherlands NR LCL,MSCI Spain NR LCL,MSCI United Kingdom NR LCL,MSCI USA NR USD
Date,,,,,,,,,,
1970-01-31,-0.031330,0.024480,0.038060,-0.048350,0.041000,-0.015471,-0.054510,0.039030,-0.007390,-0.072024
1970-02-28,-0.011542,0.044393,-0.027917,-0.029265,-0.027224,0.012680,0.008408,0.048728,-0.053596,0.055510
1970-03-31,-0.003384,0.010206,-0.011684,0.008811,0.035392,0.041257,0.013425,-0.030698,0.024452,0.005203
1970-04-30,-0.111952,-0.114342,-0.059632,-0.046194,-0.002880,-0.118091,-0.062562,-0.080656,-0.102288,-0.087283
1970-05-31,-0.064832,-0.095029,-0.027265,-0.093870,-0.094636,-0.024622,-0.024288,-0.046147,-0.050362,-0.057233


In [34]:
#get most recent 10 years of data for backtesting
from pandas.tseries.offsets import MonthEnd

backtest_start_date = max(returns.index) - MonthEnd(120) #10 years * 12 months / year = 120 months 
backtest_end_date = max(returns.index)
backtest_data = returns.loc[backtest_start_date:backtest_end_date]
training_data = returns.loc[~returns.index.isin(backtest_data.index)]

print(f"Backtest : {backtest_data.index[0].date()} : {backtest_data.index[-1].date()} | {backtest_data.shape[0]} months")
print(f"Training : {training_data.index[0].date()} : {training_data.index[-1].date()} | {training_data.shape[0]} months")
print(f"Total    : {returns.shape[0]} months | {returns.shape[1]} countries")


Backtest : 2015-12-31 : 2025-12-31 | 121 months
Training : 1970-01-31 : 2015-11-30 | 551 months
Total    : 672 months | 10 countries


### Because we need 3 months of historical data for computing covariance matrix 36 month lag splitting the training data into 2 subsets: training_data := all trianing data (1970-01-31 : 2015-11-30), train_targets_df subset of train_full 1973-01-31 : 2015-11-30

In [ ]:
train_targets_df = training_data.iloc[36:]
print(f"Training Targets : {train_targets_df.index[0].date()} : {train_targets_df.index[-1].date()} | {train_targets_df.shape[0]} months")
from dateutil.relativedelta import relativedelta
assert relativedelta(min(train_targets_df.index), min(training_data.index)).years == 3, "Check Target Returns Date Range"

Training Targets : 1973-01-31 : 2015-11-30 | 515 months


# Momentum Factors

$$Mom_{i,t} = \prod_{k}^{K}(1+r_{i,t-k}) - 1$$

In [ ]:
from pandas.tseries.offsets import MonthEnd
    
def build_momentum_factor(returns, target_dates, start_lag, end_lag, name='mom'):
    """
    Computes trailing momentum signals for each date in target_dates.

    Parameters
    ----------
    returns      : DataFrame, full return history (used for lookback windows)
    target_dates : DataFrame, dates to compute signals for
    start_lag    : int, months back for start of window (e.g. 12)
    end_lag      : int, months back for end of window (e.g. 2)
    name         : str, column name for output signal

    Returns
    -------
    lagged_cum_returns_df  : DataFrame, lagged cumulative returns per country per date (t-start_lag to t-end_lag)
    ranks_df    : DataFrame, cross-sectional rank of each country per date (1=lowest, N=highest)
    unscaled_weights_df  : DataFrame, long-short portfolio weights per country per date (sums to 0)

    """

    starts = []
    ends = []
    lagged_cum_returns = []

    for t in target_dates.index:
        start = t - MonthEnd(start_lag)
        end = t - MonthEnd(end_lag)

        assert start in returns.index, f"{start} not found in returns.index"
        assert end in returns.index, f"{end} not found in returns.index"
        
        window = returns.loc[start:end]
        signal = (1 + window).prod(axis = 0) - 1
        signal.name = t

        starts.append(start)
        ends.append(end)
        lagged_cum_returns.append(signal)

    lagged_cum_returns_df = pd.DataFrame(lagged_cum_returns)
    lagged_cum_returns_df.index.name = 'Date'
    lagged_cum_returns_df.insert(0, 'start', starts)
    lagged_cum_returns_df.insert(1, 'end', ends)

    country_cols = lagged_cum_returns_df.columns[2:]
    ranks_df = lagged_cum_returns_df[country_cols].rank(axis = 1, ascending=True, method = 'first') #ascending = True => largest mom. = rank 10 => larger weights 
    unscaled_weights_df = ranks_df.sub(ranks_df.mean(axis = 1), axis = 0)
    unscaled_weights_df = unscaled_weights_df.div(unscaled_weights_df.abs().sum(axis = 1), axis = 0)

    # raw_returns_df = pd.DataFrame(
    #     (data[country_cols] * weights_df[country_cols]).sum(axis = 1)
    # )
    # raw_returns_df.columns = [f'{name}_return']


    return lagged_cum_returns_df, ranks_df, unscaled_weights_df

In [ ]:
mom121_lagged_cum_returns_df, mom121_ranks_df, mom121_unscaled_weights_df = build_momentum_factor(training_data, train_targets_df, 12, 2, name = "mom121")


# mom121_lagged_cum_returns_df, mom121_rank_df, mom121_weights_df, raw_mom121_returns = build_momentum_factor(
#     returns, backtest_data, 12, 2, name="mom_12_1"
# )

# mom12_signals_df, mom12_rank_df, mom12_weights_df, raw_mom12_returns = build_momentum_factor(
#     returns, backtest_data, start_lag=12, end_lag=1, name="mom_12"
# )

# mom6_signals_df, mom6_rank_df, mom6_weights_df, raw_mom6_returns = build_momentum_factor(
#     returns, backtest_data, start_lag=6, end_lag=1, name="mom_6"
# )

# mom3_signals_df, mom3_rank_df, mom3_weights_df, raw_mom3_returns = build_momentum_factor(
#     returns, backtest_data, start_lag=3, end_lag=1, name="mom_3"
# )

In [49]:
# mom_compare = pd.DataFrame({
#     'mom_12_1': raw_mom121_returns.iloc[:, 0],
#     'mom_12': raw_mom12_returns.iloc[:, 0],
#     'mom_6': raw_mom6_returns.iloc[:, 0],
#     'mom_3': raw_mom3_returns.iloc[:, 0]
# })
# (1 + mom_compare).cumprod().plot(title="Momentum Factor Comparison: Growth of $1")

# def factor_summary(r):
#     ann_return = r.mean() * 12
#     ann_vol = r.std() * (12**0.5)
#     ir = ann_return / ann_vol
#     growth = (1 + r).cumprod()
#     max_dd = (growth / growth.cummax() - 1).min()
#     return pd.Series({
#         'Arithmetic Annualized Return': ann_return,
#         'Annualized Volatility': ann_vol,
#         'Information Ratio': ir,
#         'Max Drawdown': max_dd
#     })

# mom_summary = mom_compare.apply(factor_summary).T
# mom_summary


# Low Volatility Signal
For each country i and backtest month t, define trailing annualized volatility over window L as:

$$\sigma_{i,t}^{(L)} = \sqrt{(12)}*SD(r_{i,t-L},...,r_{i,t-1})$$


Then define the low-vol signal as:

$$LV_{i,t}^{(L)} = -\sigma_{i,t}^{(L)}$$




In [50]:
def build_lowvol_factor(returns, target_dates, look_back, name = 'low_vol'):
    starts = []
    ends = []
    lagged_cum_returns = []

    for t in target_dates.index:
        start = t - MonthEnd(look_back)
        end = t - MonthEnd(1)

        assert start in returns.index, f"{start} not found in returns.index"
        assert end in returns.index, f"{end} not found in returns.index"

        window = returns.loc[start:end]
        
        #lower vol ==> higher signal
        signal = -(window.std(axis = 0) * (12**0.5))
        signal.name = t 

        starts.append(start)
        ends.append(end) 
        lagged_cum_returns.append(signal)

    lagged_cum_returns_df = pd.DataFrame(lagged_cum_returns)
    lagged_cum_returns_df.index.name = 'Date'
    lagged_cum_returns_df.insert(0, 'start', starts)
    lagged_cum_returns_df.insert(1, 'end', ends)

    country_cols = lagged_cum_returns_df.columns[2:]
    rank_df = signals_df[country_cols].rank(axis=1, ascending=True, method="first")
    unscaled_weights_df = rank_df.sub(rank_df.mean(axis=1), axis=0)
    unscaled_weights_df = unscaled_weights_df.div(unscaled_weights_df.abs().sum(axis=1), axis=0)

    # raw_returns_df = pd.DataFrame(
    #     (data[country_cols] * weights_df[country_cols]).sum(axis=1)
    # )
    # raw_returns_df.columns = [f"{name}_return"]

    return signals_df, rank_df, unscaled_weights_df


        

In [ ]:
lv12_signals_df, lv12_rank_df, lv12_unscaled_weights_df = build_lowvol_factor(
    training_data, train_targets_df, look_back=12, name="lv_12"
)

# lv24_signals_df, lv24_rank_df, lv24_weights_df, raw_lv24_returns = build_lowvol_factor(
#     returns, backtest_data, look_back=24, name="lv_24"
# )

# lv36_signals_df, lv36_rank_df, lv36_weights_df, raw_lv36_returns = build_lowvol_factor(
#     returns, backtest_data, look_back=36, name="lv_36"
# )

# 50% momentum 12-1 and lowvol 12 month look back 

In [ ]:
mom121_weights_df

In [ ]:
mom121_lv12 = train_targets_df

In [ ]:
# lv_compare = pd.DataFrame({
#     'lv_12': raw_lv12_returns.iloc[:, 0],
#     'lv_24': raw_lv24_returns.iloc[:, 0],
#     'lv_36': raw_lv36_returns.iloc[:, 0]
# })

# (1 + lv_compare).cumprod().plot(
#     title="Low-Volatility Factor Comparison: Growth of $1", figsize=(10,6))

# lv_summary = lv_compare.apply(factor_summary).T
# lv_summary

# Downside Volatility / Downside Risk 

Rather than total volatility, only penallize bad months. 

For country $i$ at time $t$:

$$DD_{i,t} = \sqrt{12 * \frac{1}{L}\sum_{k=1}^{L}min(r_{i,t-k}, 0)^2}$$

Then define the signal as:

$$signal_{i,t} = -DD_{i,t}$$

- More refined than plain low vol
- Focuses on harmful volatility, not upside variation



In [ ]:
def build_downside_vol_factor(returns, backtest_data, look_back, name = 'downside_vol'):
    starts = []
    ends = []
    lagged_cum_returns = []

    for t in backtest_data.index:
        start = t - MonthEnd(look_back)
        end = t - MonthEnd(1)

        assert start in returns.index, f"{start} not found in returns.index"
        assert end in returns.index, f"{end} not found in returns.index"

        window = returns.loc[start:end]
        
        downside = window.clip(upper = 0)
        downside_dev = ((downside**2).mean(axis = 0) * 12) ** 0.5
        signal = -downside_dev
        signal.name = t

        starts.append(start)
        ends.append(end)
        lagged_cum_returns.append(signal)

    lagged_cum_returns_df = pd.DataFrame(lagged_cum_returns)
    lagged_cum_returns_df.index.name = 'Date'
    lagged_cum_returns_df.insert(0, 'start', starts)
    lagged_cum_returns_df.insert(1, 'end', ends)

    country_cols = lagged_cum_returns_df.columns[2:]
    rank_df = signals_df[country_cols].rank(axis=1, ascending=True, method="first")
    weights_df = rank_df.sub(rank_df.mean(axis=1), axis=0)
    weights_df = weights_df.div(weights_df.abs().sum(axis=1), axis=0)

    raw_returns_df = pd.DataFrame(
        (backtest_data[country_cols] * weights_df[country_cols]).sum(axis=1)
    )
    raw_returns_df.columns = [f"{name}_return"]

    return signals_df, rank_df, weights_df, raw_returns_df


        

In [ ]:
dd12_signals_df, dd12_rank_df, dd12_weights_df, raw_dd12_returns = build_downside_vol_factor(
    returns, backtest_data, 12, name='dd_12'
)

dd24_signals_df, dd24_rank_df, dd24_weights_df, raw_dd24_returns = build_downside_vol_factor(
    returns, backtest_data, 24, name='dd_24'
)

dd36_signals_df, dd36_rank_df, dd36_weights_df, raw_dd36_returns = build_downside_vol_factor(
    returns, backtest_data, 36, name='dd_36'
)

In [ ]:
dd_compare = pd.DataFrame({
    'dd_12': raw_dd12_returns.iloc[:, 0],
    'dd_24': raw_dd24_returns.iloc[:, 0],
    'dd_36': raw_dd36_returns.iloc[:, 0]
})
(1 + dd_compare).cumprod().plot(
    title="Downside Volatility Factor Comparison: Growth of $1",
    figsize=(10, 6)
)

dd_summary = dd_compare.apply(factor_summary).T
dd_summary

# Estimated Covariance Matrices 
Per Assignment Requirements:
- Covariance matrix of assets must be estimated and updated at least every month
- Each estimation must use at least 36 months of prior returns (3 years) 

Let $t$ be the backtest month, then:

$$\Sigma_t = 12 * Cov(r_{t-36}, ..., r_{t-1})$$

In [ ]:
import numpy as np 
def build_asset_cov_matrices(returns, data, look_back = 36):
    cov_dict = {}
    vol_dict = {}
    corr_dict = {}
    window_map = []

    for t in data.index:
        start = t - MonthEnd(look_back)
        end = t - MonthEnd(1)

        assert start in returns.index, f"{start} not found in returns.index"
        assert end in returns.index, f"{end} not found in returns.index"

        window = returns.loc[start:end]

        cov_t = window.cov() * 12
        vol_t = pd.Series(np.sqrt(np.diag(cov_t)), index = cov_t.index, name = t)
        corr_t = window.corr()

        cov_dict[t] = cov_t
        vol_dict[t] = vol_t
        corr_dict[t] = corr_t

        window_map.append({
            "Date": t,
            "cov_start": start,
            "cov_end": end
        })

    window_map_df = pd.DataFrame(window_map).set_index("Date")

    return cov_dict, vol_dict, corr_dict, window_map_df

In [ ]:
cov_dict, vol_dict, corr_dict, cov_window_map_df = build_asset_cov_matrices(
    returns=full_usable_data,
    data=dev_data,
    look_back=36
)

#example: first cov matrix:
# first_date = backtest_data.index[0]
# cov_dict[first_date]

# Scaling Factors to Target Volatility 
Per Assignment Requirement:
- Weights for each factor should be scaled to target 1% annual volatility

If $w_t^{raw}$ is the raw factor weight, and $\Sigma_t$ the annualized asset covariance matrix, then the raw factor volatility is:
$$\sigma_t^{raw} = \sqrt{(w_t^{raw})^T\Sigma_tw_t^{raw}}$$
The scaled weights are then:
$$w_t^{scaled} = w_t^{raw} \times \frac{0.01}{\sigma_t^{raw}}$$
Using the scaled weights enforces the factor to target 1% annual volatility for month $t$.

In [ ]:
def scale_factor_to_target_vol(weights_df, cov_dict, target_vol = 0.01):
    scaled_weights = []
    raw_factor_vols = []

    for t in weights_df.index:
        w = weights_df.loc[t]
        cov_t = cov_dict[t].loc[weights_df.columns, weights_df.columns].values

        raw_vol = np.sqrt(w.values @ cov_t @ w.values)
        raw_factor_vols.append(raw_vol)

        scaled_w = w * (target_vol / raw_vol)
        scaled_weights.append(scaled_w)

    scaled_weights_df = pd.DataFrame(scaled_weights, index = weights_df.index)
    scaled_weights_df.columns = weights_df.columns

    raw_factor_vols = pd.Series(raw_factor_vols, index = weights_df.index, name = "raw_factor_vol")

    return scaled_weights_df, raw_factor_vols
        
        

# Computing realized returns from any weights dataframe

In [ ]:
def compute_factor_returns(weights_df, dev_data, name = "factor_return"):
    country_cols = weights_df.columns
    factor_returns = (dev_data[country_cols] * weights_df[country_cols]).sum(axis = 1)
    factor_returns.name = name
    return factor_returns 

In [ ]:
# Example:
mom6_scaled_weights_df, mom6_raw_vol = scale_factor_to_target_vol(
    mom6_weights_df, cov_dict, target_vol=0.01
)

mom6_scaled_returns = compute_factor_returns(
    mom6_scaled_weights_df, backtest_data, name="mom6_scaled_return"
)

# Example:
lv36_scaled_weights_df, lv36_raw_vol = scale_factor_to_target_vol(
    lv36_weights_df, cov_dict, target_vol=0.01
)

lv36_scaled_returns = compute_factor_returns(
    lv36_scaled_weights_df, backtest_data, name="lv36_scaled_return"
)

# # Example:
# dd24_scaled_weights_df, dd24_raw_vol = scale_factor_to_target_vol(
#     dd24_weights_df, cov_dict, target_vol=0.01
# )

# dd24_scaled_returns = compute_factor_returns(
#     dd24_scaled_weights_df, backtest_data, name="dd24_scaled_return"


In [ ]:
# Example:
scaled_compare = pd.DataFrame({
    "mom_6": mom6_scaled_returns,
    "lv_36": lv36_scaled_returns
})
(1 + scaled_compare).cumprod().plot(
    title="Different Factor Comparison Growth of $1",
    figsize=(10,6)
)

scaled_summary = scaled_compare.apply(factor_summary).T
scaled_summary

In [ ]:
# (mom6_scaled_returns*0.5 + lv36_scaled_returns*0.5).plot()

# mom6_scaled_returns

# lv36_scaled_returns
# import matplotlib.pyplot as plt
# combined = mom6_scaled_returns * 0.5 + lv36_scaled_returns * 0.5
# combined.plot(title='Equally Weighted Portfolio (Mom + LV)', figsize=(12, 5))
# plt.ylabel('Return')
# plt.xlabel('Date')
# plt.show()

# combined

# (1 + combined).cumprod().plot(
#     title="Mom6 + LV36 Factor Comparison Growth of $1",
#     figsize=(10,6)
# )

# Example

In [ ]:
mom121_weights_df

In [ ]:
mom121_weights_df

In [ ]:
dev_data = returns.loc["1973-01-31":"2015-12-31"]
test_data = returns.loc["2016-01-31":"2025-12-31"]
full_usable_data = returns.loc["1970-01-31":"2025-12-31"]

#get asset covs
cov_dict, vol_dict, corr_dict, window_map_df = build_asset_cov_matrices(full_usable_data, dev_data, look_back = 36)

#get raw momentum factors weights 
mom121_lagged_cum_returns_df, mom121_rank_df, mom121_weights_df, raw_mom121_returns = build_momentum_factor(
    full_usable_data, dev_data, 12, 2, name="mom121"
)
#get scaled momentum factor weights
mom121_scaled_weights_df, mom121_raw_vol = scale_factor_to_target_vol(
    mom121_weights_df, cov_dict, target_vol=0.01
)
#get scaled momentum factor returns 
mom121_scaled_returns = compute_factor_returns(
    mom121_scaled_weights_df, dev_data, name="mom121_scaled_return"
)

#get low-volatility, look_back = 36 factor weights
lv12_signals_df, lv12_rank_df, lv12_weights_df, raw_lv12_returns = build_lowvol_factor(
    full_usable_data, dev_data, look_back=12, name="lv_12"
)
#get lv36 scaled factor weights 
lv12_scaled_weights_df, lv12_raw_vol = scale_factor_to_target_vol(
    lv12_weights_df, cov_dict, target_vol=0.01
)
#get scaled lv36 factor returns 
lv12_scaled_returns = compute_factor_returns(
    lv12_scaled_weights_df, dev_data, name="lv12_scaled_return"
)


combined = (0.5 * mom121_scaled_returns + 0.5 * lv12_scaled_returns)
(1 + combined).cumprod().plot(
    title="Mom121 + LV12 Factor Comparison Growth of $1",
    figsize=(10,6))

In [ ]:
mom121_weights_df.head()

In [ ]:
?build_asset_cov_matrices

In [ ]:
combined = (0.5*mom121_scaled_returns + 0.5*lv12_scaled_returns)
combined
(1 + combined).cumprod().plot(
    title="Mom121 + LV12 Factor Comparison Growth of $1",
    figsize=(10,6)
)